# Phase 0: YOLO11s Training Notebook

This notebook trains YOLO11s for Indian food detection.
- **Task**: detection only (no segmentation)
- **Model**: YOLO11s (small, higher accuracy than YOLOv8n)
- **Data**: IndianFoodNet (30 Indian food classes) — uploaded manually
- **Local GPU**: RTX 3050 4GB → batch=4, amp=True, workers=2
- **Kaggle GPU**: P100/T4 16GB → batch=16, workers=4 (auto-detected)

### How to upload dataset on Kaggle
1. Zip your `indianfoodnet_yolo/` folder → `indianfoodnet_yolo.zip`
2. Go to **Kaggle → Datasets → New Dataset**
3. Upload the zip, name it `indianfoodnet-yolo`, publish (can be private)
4. In this notebook → **Add Data** (right panel) → search `indianfoodnet-yolo` → attach
5. Dataset will appear at `/kaggle/input/indianfoodnet-yolo/`

> **Run in VS Code** (local) or **Kaggle Notebook** (recommended for training).


## Cell 00 — Install Dependencies

In [1]:
%pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 125.1 kB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


## Cell 0 — Environment Detection & Dataset Path Setup

In [2]:
# Cell 0: Environment detection + dataset path resolution (no downloads)
import os
from pathlib import Path

KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
COLAB  = 'google.colab' in str(globals().get('__builtins__', ''))
CLOUD  = KAGGLE or COLAB

print('=== Environment ===')
print(f'Running on Kaggle : {KAGGLE}')
print(f'Running on Colab  : {COLAB}')
print(f'Cloud mode        : {CLOUD}')

# ── Resolve dataset path ───────────────────────────────────────────────────
if KAGGLE:
    kaggle_input = Path('/kaggle/input')
    matches = list(kaggle_input.rglob('data.yaml'))
    if not matches:
        raise FileNotFoundError(
            'data.yaml not found under /kaggle/input/\n'
            'Make sure you attached the indianfoodnet-yolo dataset to this notebook.\n'
            'Kaggle → Add Data (right panel) → search indianfoodnet-yolo → attach'
        )
    DATA_YAML    = str(matches[0])
    DATASET_ROOT = str(matches[0].parent)
    print(f'\n✅ Dataset found at: {DATA_YAML}')

elif COLAB:
    DATASET_ROOT = 'data/indianfoodnet_yolo'
    DATA_YAML    = f'{DATASET_ROOT}/data.yaml'
    if not Path(DATA_YAML).exists():
        raise FileNotFoundError(
            f'data.yaml not found at {DATA_YAML}\n'
            'Upload indianfoodnet_yolo/ to Colab or mount Google Drive.'
        )
    print(f'\n✅ Dataset found at: {DATA_YAML}')

else:
    DATASET_ROOT = 'data/indianfoodnet_yolo'
    DATA_YAML    = f'{DATASET_ROOT}/data.yaml'
    if not Path(DATA_YAML).exists():
        raise FileNotFoundError(
            f'data.yaml not found at {DATA_YAML}\n'
            'Download IndianFoodNet from Roboflow and place it at data/indianfoodnet_yolo/'
        )
    print(f'\n✅ Dataset found at: {DATA_YAML}')

print(f'Dataset root : {DATASET_ROOT}')
print(f'data.yaml    : {DATA_YAML}')


=== Environment ===
Running on Kaggle : True
Running on Colab  : False
Cloud mode        : True

✅ Dataset found at: /kaggle/input/datasets/jaspreetjtsingh/indian-food-images/data.yaml
Dataset root : /kaggle/input/datasets/jaspreetjtsingh/indian-food-images
data.yaml    : /kaggle/input/datasets/jaspreetjtsingh/indian-food-images/data.yaml


## Cell 1 — GPU Check & Imports

In [3]:
# Cell 1: GPU check + imports
import os          # FIX: explicit import — required if this cell runs standalone
import torch
import yaml
import json
import shutil
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# Re-detect environment in case cells run out of order
KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
COLAB  = 'google.colab' in str(globals().get('__builtins__', ''))
CLOUD  = KAGGLE or COLAB

print('=== GPU Check ===')
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA version    : {torch.version.cuda}')
    print(f'GPU device      : {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU VRAM        : {vram_gb:.1f} GB')
    if vram_gb < 6:
        print('⚠️  <6GB VRAM — forcing local config (batch=4, workers=2)')
        CLOUD = False
else:
    print('⚠️  WARNING: CUDA not available — training will be extremely slow!')

# FIX: dynamic device — falls back to CPU if CUDA not available
DEVICE  = 0 if torch.cuda.is_available() else 'cpu'

# Set working directory to project root (local only; Kaggle runs from /kaggle/working)
if not CLOUD:
    project_root = Path.cwd()
    while not (project_root / 'data').exists() and project_root != project_root.parent:
        project_root = project_root.parent
    os.chdir(project_root)
    print(f'Working directory: {project_root}')

# Training config based on hardware
BATCH   = 16 if CLOUD else 4
WORKERS = 4  if CLOUD else 2
print(f'\nConfig → device={DEVICE}, batch={BATCH}, workers={WORKERS}, cloud={CLOUD}')


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
=== GPU Check ===
PyTorch version : 2.10.0+cu128
CUDA available  : True
CUDA version    : 12.8
GPU device      : Tesla T4
GPU VRAM        : 15.6 GB

Config → device=0, batch=16, workers=4, cloud=True


## Cell 2 — YOLO11s Detection Training

In [4]:
# Cell 2: YOLO11s detection training
# DATA_YAML and DEVICE are set in Cell 0/1 — run those cells first
print('=== Starting YOLO11s Training ===')
print(f'Dataset : {DATA_YAML}')
print(f'Device  : {DEVICE} | Batch : {BATCH} | Workers: {WORKERS} | Cloud: {CLOUD}')

# NOTE: correct model name is 'yolo11s.pt' (no 'v'), not 'yolov11s.pt'
model = YOLO('yolo11s.pt')

training_args = {
    'data'        : DATA_YAML,   # resolved in Cell 0 — works on Kaggle + local
    'task'        : 'detect',
    'epochs'      : 100,
    'imgsz'       : 640,
    'batch'       : BATCH,       # 16 on Kaggle P100/T4, 4 on local 4GB
    'patience'    : 20,
    'device'      : DEVICE,      # FIX: dynamic — 0 (GPU) or 'cpu' (fallback)
    'workers'     : WORKERS,
    'amp'         : True,        # mandatory — cuts VRAM by ~40%
    'save_period' : 10,
    'project'     : 'models/runs',
    'name'        : 'yolo11s_indian',
}

print(f'Training args: {training_args}')
results = model.train(**training_args)
print('\n✅ YOLO11s training completed!')


=== Starting YOLO11s Training ===
Dataset : /kaggle/input/datasets/jaspreetjtsingh/indian-food-images/data.yaml
Device  : 0 | Batch : 16 | Workers: 4 | Cloud: True
Training args: {'data': '/kaggle/input/datasets/jaspreetjtsingh/indian-food-images/data.yaml', 'task': 'detect', 'epochs': 100, 'imgsz': 640, 'batch': 16, 'patience': 20, 'device': 0, 'workers': 4, 'amp': True, 'save_period': 10, 'project': 'models/runs', 'name': 'yolo11s_indian'}
Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/jaspreetjtsingh/indian-food-images/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, e

## Cell 3 — Copy best.pt → models/yolo11s_indian.pt

In [11]:
# Cell 3: Copy best.pt → models/yolo11s_indian.pt
print('=== Saving YOLO11s Model ===')

best_model_path = Path('models/runs/yolo11s_indian/weights/best.pt')

if not best_model_path.exists():
    candidates = sorted(
        Path('models/runs').glob('yolo11s_indian*/weights/best.pt'),
        key=lambda p: p.stat().st_mtime, reverse=True
    )
    if candidates:
        best_model_path = candidates[0]
        print(f'Found via scan: {best_model_path}')
    else:
        raise FileNotFoundError('best.pt not found — did Cell 2 training complete?')

dest = Path('models/yolo11s_indian.pt')
dest.parent.mkdir(exist_ok=True)
shutil.copy2(best_model_path, dest)

print(f'✅ Saved : {dest}')
print(f'   From  : {best_model_path}')
print(f'   Size  : {dest.stat().st_size / 1e6:.1f} MB')
print('→ Update .env: YOLO_MODEL_PATH=models/yolo11s_indian.pt')

if KAGGLE:
    import shutil as _sh
    kaggle_out = Path('/kaggle/working/yolo11s_indian.pt')
    _sh.copy2(dest, kaggle_out)
    print(f'\n✅ Also copied to Kaggle output: {kaggle_out}')


=== Saving YOLO11s Model ===


AttributeError: 'str' object has no attribute 'exists'

## Cell 4 — Save Class Names → models/class_names.json

In [ ]:
# Cell 4: Save class names → models/class_names.json
print('=== Extracting Class Names ===')

# DATA_YAML resolved in Cell 0
with open(DATA_YAML, 'r') as f:
    data_config = yaml.safe_load(f)

class_names = data_config.get('names', [])
print(f'Found {len(class_names)} classes:')
for i, name in enumerate(class_names):
    print(f'  {i:2d}: {name}')

out_path = Path('models/class_names.json')
out_path.parent.mkdir(exist_ok=True)
with open(out_path, 'w') as f:
    json.dump(class_names, f, indent=2)
print(f'\n✅ Saved to: {out_path}')

# Cross-check against expected 30 classes
expected = [
    'biryani','butter_chicken','chapati','chole_bhature','dal_makhani',
    'dal_tadka','dosa','gulab_jamun','idli','jalebi','kadai_paneer',
    'kathi_roll','kheer','kulfi','masala_dosa','medu_vada','naan',
    'pakoda','palak_paneer','paneer_butter_masala','pav_bhaji','poha',
    'puri','rasgulla','ras_malai','samosa','shahi_paneer','uttapam',
    'vada_pav','momos'
]
normalised = [n.lower().replace(' ', '_') for n in class_names]
missing = [c for c in expected if c not in normalised]
if missing:
    print(f'\n⚠️  Expected classes missing from dataset: {missing}')
else:
    print('\n✅ All 30 expected food classes present in dataset')

if KAGGLE:
    import shutil as _sh
    _sh.copy2(out_path, '/kaggle/working/class_names.json')
    print('✅ Copied class_names.json to Kaggle output panel')


## Cell 5 — Plot Training Loss Curves

In [ ]:
# Cell 5: Plot training curves inline
print('=== Plotting Training Curves ===')

# Find results.csv
results_csv = Path('models/runs/yolo11s_indian/results.csv')
if not results_csv.exists():
    candidates = sorted(
        Path('models/runs').glob('yolo11s_indian*/results.csv'),
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )
    if candidates:
        results_csv = candidates[0]
    else:
        raise FileNotFoundError('results.csv not found — did Cell 2 complete?')

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()  # strip whitespace Ultralytics sometimes adds
print(f'Loaded {len(df)} epochs | columns: {df.columns.tolist()}')

# FIX: expanded to 2x3 grid — added DFL Loss panel
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('YOLO11s Training — Indian Food Detection', fontsize=15, fontweight='bold')

axes[0,0].plot(df['epoch'], df['train/box_loss'], label='Train', color='#2196F3')
axes[0,0].plot(df['epoch'], df['val/box_loss'],   label='Val',   color='#FF5722', linestyle='--')
axes[0,0].set_title('Box Loss'); axes[0,0].legend(); axes[0,0].set_xlabel('Epoch')

axes[0,1].plot(df['epoch'], df['train/cls_loss'], label='Train', color='#2196F3')
axes[0,1].plot(df['epoch'], df['val/cls_loss'],   label='Val',   color='#FF5722', linestyle='--')
axes[0,1].set_title('Classification Loss'); axes[0,1].legend(); axes[0,1].set_xlabel('Epoch')

# FIX: DFL Loss — bounding box quality indicator (was missing before)
axes[0,2].plot(df['epoch'], df['train/dfl_loss'], label='Train', color='#2196F3')
axes[0,2].plot(df['epoch'], df['val/dfl_loss'],   label='Val',   color='#FF5722', linestyle='--')
axes[0,2].set_title('DFL Loss (Box Quality)'); axes[0,2].legend(); axes[0,2].set_xlabel('Epoch')

axes[1,0].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', color='#4CAF50')
axes[1,0].plot(df['epoch'], df['metrics/recall(B)'],    label='Recall',    color='#FF9800')
axes[1,0].set_title('Precision & Recall'); axes[1,0].legend(); axes[1,0].set_xlabel('Epoch')

axes[1,1].plot(df['epoch'], df['metrics/mAP50(B)'],    label='mAP@0.5',      color='#9C27B0')
axes[1,1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', color='#607D8B', linestyle='--')
axes[1,1].set_title('Mean Average Precision'); axes[1,1].legend(); axes[1,1].set_xlabel('Epoch')

# Bonus: val box+cls+dfl total loss estimate
df['val/total_loss'] = df['val/box_loss'] + df['val/cls_loss'] + df['val/dfl_loss']
axes[1,2].plot(df['epoch'], df['val/total_loss'], label='Val Total Loss', color='#E91E63')
axes[1,2].set_title('Val Total Loss (box+cls+dfl)'); axes[1,2].legend(); axes[1,2].set_xlabel('Epoch')

plt.tight_layout()
plot_path = 'models/training_curves_yolo11s.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Plot saved to {plot_path}')

if KAGGLE:
    import shutil as _sh
    _sh.copy2(plot_path, '/kaggle/working/training_curves_yolo11s.png')
    print('✅ Copied plot to Kaggle output panel')

# Final metrics summary
final      = df.iloc[-1]
best_epoch = df['metrics/mAP50(B)'].idxmax()
best_map50 = df['metrics/mAP50(B)'].max()
print(f'\n=== Final Metrics ===')
print(f'Epochs trained       : {len(df)}')
print(f'Best mAP@0.5         : {best_map50:.4f} (epoch {best_epoch})')
print(f'Final mAP@0.5        : {final["metrics/mAP50(B)"]:.4f}')
print(f'Final mAP@0.5:0.95   : {final["metrics/mAP50-95(B)"]:.4f}')
print(f'Final Precision      : {final["metrics/precision(B)"]:.4f}')
print(f'Final Recall         : {final["metrics/recall(B)"]:.4f}')
